In [1]:
import sys

sys.path.append("../")
import concurrent.futures

import xarray as xr

from visualisation import *

In [2]:
df_address = pd.read_csv("Adelide_postcode_points.csv")

In [2]:
bom_path = "/mnt/d/bom_nci/2023/**/**/"
files = glob(bom_path + "*.nc")
len(files)

37591

In [3]:
def process_file(file):
    return (
        xr.open_dataset(file)
        .to_dataframe()
        .reset_index(drop=False)
        .merge(df_address, on=["latitude", "longitude"], how="inner")
    )


num_cores = 10

In [15]:
with concurrent.futures.ProcessPoolExecutor(max_workers=num_cores) as executor:
    df_list = list(executor.map(process_file, files))

df = pd.concat(df_list, axis=0).reset_index(drop=True)
df = df.dropna(subset="direct_normal_irradiance").reset_index(drop=True)

In [16]:
df.to_csv("/mnt/d/bom_nci/2023/NCI_processed_Adelide.csv", index=False)

In [17]:
os.path.getsize("/mnt/d/bom_nci/2023/NCI_processed_Adelide.csv") / (1024 * 1024)

1042.3862199783325

In [2]:
df = pd.read_csv("/mnt/d/bom_nci/2023/NCI_processed_Adelide.csv")

In [10]:
df.drop(columns=["latitude", "longitude", "crs", "julian_date"], inplace=True)

In [12]:
df[:2]

,time,surface_global_irradiance,direct_normal_irradiance,surface_diffuse_irradiance,quality_mask,cloud_type,cloud_optical_depth,solar_elevation,solar_azimuth,postcode
0,2023-10-31 23:50:00,829.60,965.16,88.42,1.0,0.0,0.0,50.2,68.3,5044.0
1,2023-10-31 23:50:00,830.71,966.76,88.11,1.0,0.0,0.0,50.2,68.3,5046.0


In [15]:
df.groupby(["time", "postcode"]).mean().reset_index().to_csv(
    "/mnt/d/bom_nci/2023/NCI_processed_Adelaide_grouped.csv", index=False
)

In [16]:
df = pd.read_csv("/mnt/d/bom_nci/2023/NCI_processed_Adelaide_grouped.csv")

In [17]:
df

,time,postcode,surface_global_irradiance,direct_normal_irradiance,surface_diffuse_irradiance,quality_mask,cloud_type,cloud_optical_depth,solar_elevation,solar_azimuth
0,2022-12-31 19:30:00,1291.0,0.000000,0.0,0.000000,1.0,0.000000,0.000000,-0.200000,118.7
1,2022-12-31 19:30:00,1361.0,0.000000,0.0,0.000000,1.0,0.000000,0.000000,-0.200000,118.7
2,2022-12-31 19:30:00,5000.0,0.000000,0.0,0.000000,1.0,0.000000,0.000000,-0.300000,118.8
3,2022-12-31 19:30:00,5007.0,0.000000,0.0,0.000000,1.0,0.000000,0.000000,-0.300000,118.8
4,2022-12-31 19:30:00,5008.0,0.000000,0.0,0.000000,1.0,0.000000,0.000000,-0.300000,118.8
...,...,...,...,...,...,...,...,...,...,...
2720491,2023-12-31 09:50:00,5245.0,0.000000,0.0,0.000000,1.0,0.000000,0.000000,-0.100000,241.3
2720492,2023-12-31 09:50:00,5350.0,0.000000,0.0,0.000000,1.0,0.000000,0.000000,-0.300000,241.3
2720493,2023-12-31 09:50:00,5501.0,0.000000,0.0,0.000000,1.0,3.666667,4.973333,-0.146667,241.4
2720494,2023-12-31 09:50:00,5950.0,0.066667,0.0,0.066667,1.0,0.000000,0.000000,0.000000,241.4
